# Week 6 — Image Segmentation for Object-Based Image Analysis (GEOBIA)
**Geoprocesamiento · Maestría en Geomática · Universidad Nacional de Colombia**

## What this notebook does
In this notebook you will build, step by step, a complete **object-based image analysis (GEOBIA)** workflow with Sentinel-2 imagery. Instead of analysing every pixel on its own, we will first group neighbouring, similar pixels into **image objects** (segments) and then describe and analyse those objects.

The workflow reuses every cloud-native format from Week 5:

```
STAC (find the scene) ──► COG (download only our area) ──► segmentation ──► objects + attributes ──► GeoParquet
```

**Base example:** flower greenhouses in the Sabana de Bogotá (Madrid – Funza – Mosquera). Greenhouses have plastic roofs that look very different from the surrounding pastures, and they have regular shapes, so they are a good object to start with.

## What you will learn
1. How to find and stream a Sentinel-2 scene with **STAC + COG** (Week 5 recap).
2. How two segmentation algorithms work in practice: **SLIC** and **Felzenszwalb**.
3. How the **scale parameter** controls the size of the objects (over- vs. under-segmentation).
4. How to choose parameters with an objective metric (**Johnson & Xie, 2011**).
5. How to turn segments into **polygons with attributes** and save them as **GeoParquet**.
6. A first look at grouping objects with **k-means** (preview of next week).

## How to use this notebook
- Read the text cell **before** each code cell: it explains what the code does and what you should see.
- 👉 The **only** cell you must edit to work on your own area is **CONFIGURE YOUR AREA OF INTEREST**.
- Cells marked **🔧 Try it** invite you to change a value and run again. Experimenting is part of the exercise.

## Notebook map
| Stage | What happens | Main tool |
|---|---|---|
| 0 | Setup and configuration | — |
| 1 | Search for a Sentinel-2 scene | `pystac-client` |
| 2 | Stream the bands, mask clouds, compute NDVI | `rioxarray` |
| 3 | Segment the image (SLIC and Felzenszwalb) | `scikit-image` |
| 4 | Evaluate the segmentation and choose parameters | NumPy / pandas |
| 5 | Compute attributes for each object | pandas / `scikit-image` |
| 6 | Convert segments into polygons | `rasterio` / `geopandas` |
| 7 | Save as GeoParquet and compare with GeoJSON | `geopandas` / `pyarrow` |
| 8 | Group objects with k-means and map them | `scikit-learn` / `folium` |

---
# Stage 0 · Setup

## 0.1 Install the libraries (only the first time)
If you have never used some of these libraries in your environment, remove the `#` at the beginning of the line below and run the cell once. Then add the `#` back so you do not reinstall them every time.

| Library | What we use it for |
|---|---|
| `pystac-client` | Search the STAC catalogue (Week 5) |
| `rioxarray` | Read Cloud-Optimised GeoTIFFs over the internet (Week 5) |
| `scikit-image` | Segmentation algorithms (SLIC, Felzenszwalb) |
| `scikit-learn` | k-means and standardisation |
| `geopandas`, `pyarrow` | Vector data and GeoParquet (Week 5) |
| `folium`, `mapclassify` | Interactive maps with `.explore()` |
| `matplotlib` | Figures |

In [ ]:
# Remove the '#' below to install the libraries, run the cell once, then put the '#' back.
# !pip install pystac-client rioxarray scikit-image scikit-learn geopandas pyarrow folium mapclassify matplotlib

## 0.2 Import the libraries
Here we load every tool used in the notebook. Each import has a short comment saying what it is for, so you can come back here if you see a function you do not recognise later on.

**What you should see:** nothing. If the cell runs without errors, all libraries are installed. If you get `ModuleNotFoundError`, go back to cell 0.1.

In [ ]:
# --- Python standard library --------------------------------------------------
import os                      # create folders and check file sizes
import time                    # measure how long each step takes
import warnings                # hide non-critical warnings to keep the output clean

# --- Data handling --------------------------------------------------------------
import numpy as np             # arrays: every band is a 2-D NumPy array of numbers
import pandas as pd            # tables: we store one row per object
import matplotlib.pyplot as plt  # figures

# --- Geospatial -----------------------------------------------------------------
import geopandas as gpd        # vector data (polygons) + GeoParquet
import rioxarray               # opens COGs as labelled arrays with CRS information
from pystac_client import Client          # connects to a STAC API (Week 5)
from rasterio.enums import Resampling      # methods to change pixel size (bilinear, nearest)
from rasterio.features import shapes       # converts a raster of labels into polygons
from shapely.geometry import box, shape    # build geometries (box = rectangle from a bbox)

# --- Image analysis and machine learning ----------------------------------------
from skimage.segmentation import slic, felzenszwalb, mark_boundaries  # segmentation
from skimage.measure import regionprops_table   # shape measurements for each segment
from sklearn.cluster import KMeans              # k-means clustering
from sklearn.preprocessing import StandardScaler  # put attributes on the same scale

warnings.filterwarnings("ignore")   # hide warnings (errors will still be shown)
print("✓ All libraries imported")

## 0.3 👉 CONFIGURE YOUR AREA OF INTEREST
This is the cell you will change to work on **your own area**. Each variable is explained below:

| Variable | Meaning | Tips |
|---|---|---|
| `AOI_NAME` | A short name for your area, used to name the output files | No spaces or accents, e.g. `palma_meta` |
| `BBOX` | The rectangle of your area: `[min_lon, min_lat, max_lon, max_lat]` in degrees (EPSG:4326) | Get it from [bboxfinder.com](http://bboxfinder.com) or [geojson.io](https://geojson.io). Keep it around **10 × 10 km** (≈ 0.1° × 0.1° near the equator) |
| `DATE_RANGE` | Period to search for images, `"start/end"` | Choose the dry season of your region to find images with fewer clouds |
| `MAX_CLOUD` | Maximum cloud cover of the scene, in % | Increase it (30, 50) if no scenes are found |
| `OBJECT_OF_INTEREST` | What you want to delineate | It only documents your choice; think about its size compared with a 10 m pixel |

**Why only 10 × 10 km?** That is about 1 000 × 1 000 pixels. Segmentation is repeated many times in Stage 4, so a bigger area would make the notebook very slow or run out of memory.

In [ ]:
AOI_NAME   = "sabana_greenhouses"           # used to name the output folder and files
BBOX       = [-74.30, 4.70, -74.20, 4.80]   # [min_lon, min_lat, max_lon, max_lat] in degrees
DATE_RANGE = "2025-12-01/2026-03-31"        # December–March = dry season in the Sabana de Bogotá
MAX_CLOUD  = 20                              # keep scenes with less than 20 % cloud cover
OBJECT_OF_INTEREST = "greenhouses"           # what are YOU trying to delineate?

# Create a folder for the results (exist_ok=True: do not fail if it already exists)
OUTPUT_DIR = f"outputs_{AOI_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Quick sanity check of the bounding box size (1 degree ≈ 111 km near the equator)
width_km  = (BBOX[2] - BBOX[0]) * 111
height_km = (BBOX[3] - BBOX[1]) * 111
print(f"✓ Area '{AOI_NAME}': about {width_km:.0f} km × {height_km:.0f} km")
print(f"✓ Results will be saved in the folder: {OUTPUT_DIR}/")

---
# Stage 1 · Find a Sentinel-2 scene with STAC (Week 5 recap)

Remember from Week 5: a **STAC API** is a searchable catalogue. Each **Item** is one satellite scene (a GeoJSON Feature with a date), and its **Assets** are links to the actual files, here one **COG per band**.

We will use the public **Earth Search** catalogue (Element 84, on AWS) and the collection **`sentinel-2-l2a`**: Sentinel-2 images already corrected to surface reflectance (Level-2A), which is what we need for analysis.

## 1.1 Search the catalogue
We ask the catalogue for scenes that:
- belong to the `sentinel-2-l2a` collection,
- intersect our `BBOX`,
- were acquired within `DATE_RANGE`,
- have a cloud cover (`eo:cloud_cover`) lower than `MAX_CLOUD`.

**What you should see:** the number of scenes found. Nothing is downloaded yet: we only receive metadata (JSON).

In [ ]:
# Connect to the Earth Search STAC API (this only reads the catalogue description)
catalog = Client.open("https://earth-search.aws.element84.com/v1")

# Build the search. The API does the filtering on the server side.
search = catalog.search(
    collections=["sentinel-2-l2a"],                 # Sentinel-2 surface reflectance
    bbox=BBOX,                                      # our rectangle, in degrees
    datetime=DATE_RANGE,                            # our period
    query={"eo:cloud_cover": {"lt": MAX_CLOUD}},    # 'lt' = less than
    max_items=100,                                  # do not return more than 100 scenes
)

# .items() returns the STAC Items; list() collects them so we can count and sort them
items = list(search.items())
print(f"✓ {len(items)} scenes found with less than {MAX_CLOUD} % clouds")

## 1.2 Keep only scenes that fully cover our area, and pick the clearest
Sentinel-2 images are delivered in **tiles of 110 × 110 km**. If our area falls on the edge of a tile, a scene might cover only part of it, and we would get stripes with no data. So we keep only the scenes whose footprint (`item.geometry`) **completely contains** our rectangle.

Then we sort them by cloud cover and choose the first one (the least cloudy).

⚠️ `eo:cloud_cover` is the cloud cover of the **whole 110 × 110 km scene**, not of our small area. A scene with 15 % clouds may be completely clear over our area, or completely cloudy. That is why in Stage 2 we will also apply a pixel-by-pixel cloud mask.

**What you should see:** a small table with the candidate scenes and the ID of the selected one.

In [ ]:
# Our area as a Shapely polygon, so we can compare it with each scene footprint
aoi_geom = box(*BBOX)          # box(min_lon, min_lat, max_lon, max_lat) → rectangle

# Keep the scenes whose footprint contains the whole area
items = [it for it in items if shape(it.geometry).contains(aoi_geom)]

# Sort from least to most cloudy
items = sorted(items, key=lambda it: it.properties["eo:cloud_cover"])

if not items:
    raise RuntimeError("No scenes found: widen DATE_RANGE, raise MAX_CLOUD or move/shrink BBOX.")

# Show the candidates as a table (first 10)
candidates = pd.DataFrame([{
    "scene_id": it.id,
    "date": it.datetime.strftime("%Y-%m-%d"),
    "cloud_%": round(it.properties["eo:cloud_cover"], 1),
} for it in items])
display(candidates.head(10))

# Select the least cloudy scene.
# 🔧 If your area turns out to be cloudy in Stage 2, come back and use items[1], items[2], ...
item = items[0]
print(f"✓ Selected scene: {item.id} | {item.datetime:%Y-%m-%d} | cloud {item.properties['eo:cloud_cover']:.1f} %")

## 1.3 Look at the assets (files) of the selected scene
Each asset is one file in the cloud. The ones we will use are:

| Asset key | Sentinel-2 band | Pixel size | Why we use it |
|---|---|---|---|
| `blue` | B02 | 10 m | Plastic roofs are bright in the blue |
| `green` | B03 | 10 m | True-colour image |
| `red` | B04 | 10 m | True-colour image and NDVI |
| `nir` | B08 | 10 m | Vegetation is very bright in the near infrared (NDVI) |
| `swir16` | B11 | 20 m | Separates bare soil, buildings and water |
| `scl` | Scene Classification | 20 m | Pixel-by-pixel mask of clouds and shadows |

**What you should see:** the address (URL) of each COG. These are the files we will stream in Stage 2.

In [ ]:
for key in ["blue", "green", "red", "nir", "swir16", "scl"]:
    asset = item.assets[key]
    print(f"{key:7s} → {asset.href}")

---
# Stage 2 · Stream the bands, mask clouds and compute NDVI

## 2.1 A function to read one band of our area
We write a small function, `read_asset()`, that we will call once per band. It does three things:

1. **Opens the COG** over the internet with `rioxarray.open_rasterio()`. At this point only the header is read (Week 5: *header inspection*).
2. **Cuts our area** with `clip_box()`. Our `BBOX` is in degrees but the image is in UTM (metres); with `crs="EPSG:4326"` rioxarray converts the box for us. `.load()` then downloads **only the tiles that cover our area**.
3. **Converts digital numbers into reflectance.** The COG stores integers (e.g. 1 543). Reflectance is a value between 0 and 1. Earth Search tells us the `scale` and `offset` in the asset metadata (`raster:bands`): `reflectance = DN × scale + offset`. Since 2022 the offset is −0.1, so ignoring it would bias NDVI.

The `scl` band is not reflectance but **class codes**, so we do not convert it.

In [ ]:
def read_asset(item, key, scale_reflectance=True):
    '''Read one band (asset) of a STAC item, clipped to BBOX.

    item : the STAC Item selected in Stage 1
    key  : the asset name, e.g. "red" or "nir"
    scale_reflectance : True for spectral bands, False for class layers (scl)
    returns: an xarray DataArray (2-D array + coordinates + CRS)
    '''
    asset = item.assets[key]

    # 1) Open the COG: only the header (metadata) is read here, no pixels yet.
    #    masked=True turns the no-data value into NaN (Not a Number).
    #    squeeze("band") removes the extra "band" dimension (each file has one band).
    da = rioxarray.open_rasterio(asset.href, masked=True).squeeze("band", drop=True)

    # 2) Clip to our area. The bbox is in degrees (EPSG:4326) and is converted to the
    #    image CRS automatically. .load() downloads ONLY the tiles inside the bbox.
    da = da.rio.clip_box(*BBOX, crs="EPSG:4326").load()

    # 3) Digital numbers → surface reflectance (0–1), using the scale/offset in the metadata
    if scale_reflectance:
        band_info = (asset.extra_fields.get("raster:bands") or [{}])[0]
        scale  = band_info.get("scale", 0.0001)   # default values if the metadata is missing
        offset = band_info.get("offset", 0.0)
        da = da * scale + offset
    return da

print("✓ Function read_asset() defined")

## 2.2 Read the bands
Now we call `read_asset()` for each band. The 10 m bands share the same grid. The 20 m bands (`swir16` and `scl`) have pixels twice as large, so we **resample** them onto the 10 m grid of the red band with `reproject_match()`:

- **Bilinear** for `swir16`: it is a continuous value, so averaging neighbours makes sense.
- **Nearest neighbour** for `scl`: it contains class codes (e.g. 8 = cloud). Averaging 8 and 4 would give 6 = "water", which is nonsense. Nearest neighbour keeps the original codes.

**What you should see:** the time it took, the size of the grid (rows, columns) and the CRS (a UTM zone, e.g. EPSG:32618). This may take from a few seconds to a minute, depending on your connection.

In [ ]:
t0 = time.time()

# The red band defines the reference 10 m grid for everything else
red = read_asset(item, "red")

bands = {
    "blue":  read_asset(item, "blue"),
    "green": read_asset(item, "green"),
    "red":   red,
    "nir":   read_asset(item, "nir"),
    # 20 m band → resampled to the 10 m grid of 'red' with bilinear interpolation
    "swir16": read_asset(item, "swir16").rio.reproject_match(red, resampling=Resampling.bilinear),
}
# Scene Classification Layer (class codes) → nearest neighbour, no reflectance scaling
scl = read_asset(item, "scl", scale_reflectance=False).rio.reproject_match(red, resampling=Resampling.nearest)

print(f"✓ Streamed {len(bands) + 1} bands in {time.time() - t0:.1f} s")
print(f"✓ Grid size: {red.shape[0]} rows × {red.shape[1]} columns | CRS: {red.rio.crs}")

# Save two pieces of georeferencing we will need in Stage 6:
TRANSFORM = red.rio.transform()   # affine transform: converts (row, column) → (x, y) in metres
CRS = red.rio.crs                 # coordinate reference system of the image (UTM)
PIXEL_AREA = abs(TRANSFORM.a * TRANSFORM.e)   # pixel width × height = 10 m × 10 m = 100 m²

## 2.3 Build a cloud mask and compute NDVI
**Cloud mask.** The `scl` layer classifies every pixel. We mark as **invalid** the pixels that belong to these classes:

| Code | Class | | Code | Class |
|---|---|---|---|---|
| 0 | No data | | 8 | Cloud, medium probability |
| 1 | Saturated or defective | | 9 | Cloud, high probability |
| 3 | Cloud shadow | | 10 | Thin cirrus |

`valid` is a True/False array of the same size as the image: **True = usable pixel**.

⚠️ Plastic greenhouse roofs are very bright and the SCL sometimes confuses them with clouds (classes 8 or 9). If you see greenhouses disappearing later, this is the reason. 🔧 You can test removing `8` from `BAD_SCL` and compare.

**NDVI** (Normalised Difference Vegetation Index) = (NIR − Red) / (NIR + Red). It is close to 0.6–0.9 for dense vegetation and close to 0 (or negative) for plastic, buildings, bare soil and water. We add `1e-9` to the denominator to avoid dividing by zero.

**What you should see:** the percentage of valid pixels. If it is very low (< 50 %), go back to Stage 1.2 and choose another scene.

In [ ]:
BAD_SCL = [0, 1, 3, 8, 9, 10]   # SCL classes we do not trust (see table above)

# Convert the DataArrays into plain NumPy arrays of 32-bit floats (smaller and faster)
arrays = {name: da.values.astype("float32") for name, da in bands.items()}

# A pixel is valid if: its SCL class is not in BAD_SCL  AND  no band is NaN
valid = ~np.isin(scl.values, BAD_SCL)
for name, a in arrays.items():
    valid &= np.isfinite(a)          # &= means "keep True only if this band is also OK"

# NDVI from the near-infrared and red reflectance
arrays["ndvi"] = (arrays["nir"] - arrays["red"]) / (arrays["nir"] + arrays["red"] + 1e-9)

print(f"✓ Valid pixels: {valid.mean():.1%} of the area")
print(f"✓ NDVI range over valid pixels: {np.nanmin(arrays['ndvi'][valid]):.2f} to {np.nanmax(arrays['ndvi'][valid]):.2f}")

## 2.4 Look at the data
Before segmenting, always **look at your image**. We plot:
- **True colour** (red, green, blue bands), as the human eye would see it. Reflectance values are small (0–0.3), so we apply a **contrast stretch**: values between the 2nd and 98th percentiles are spread over the full 0–1 range.
- **NDVI** with a red-yellow-green colour scale. Invalid pixels are shown in white.

**What you should see:** in the Sabana example, greenhouses appear as bright white/grey rectangles in true colour and as red/orange (low NDVI) patches surrounded by green pastures.

In [ ]:
def stretch(a, valid):
    '''Contrast stretch: map the 2nd–98th percentile of valid pixels to 0–1.'''
    lo, hi = np.nanpercentile(a[valid], [2, 98])
    return np.clip((a - lo) / (hi - lo), 0, 1)     # clip keeps values inside 0–1

# Stack red, green and blue as the three colour channels of an RGB image (rows, cols, 3)
rgb = np.dstack([stretch(arrays[b], valid) for b in ("red", "green", "blue")])

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].imshow(rgb)
ax[0].set_title("True colour (B4-B3-B2)")

ndvi_show = np.where(valid, arrays["ndvi"], np.nan)          # hide invalid pixels
im = ax[1].imshow(ndvi_show, cmap="RdYlGn", vmin=-0.2, vmax=0.9)
ax[1].set_title("NDVI")
plt.colorbar(im, ax=ax[1], shrink=0.7, label="NDVI")

for a in ax:
    a.axis("off")      # hide the pixel-number axes
plt.tight_layout()
plt.show()

---
# Stage 3 · Segmentation: from pixels to image objects

**Segmentation** groups neighbouring pixels that are similar into **segments** (image objects). The result is a **label image**: an array of the same size as our image where each pixel holds the number of the segment it belongs to (1, 2, 3, …). We use **0 for invalid pixels** (clouds, no data).

```
Image (reflectance)            Label image
0.21 0.22 0.05 0.04            1 1 2 2
0.20 0.23 0.06 0.05     →      1 1 2 2
0.04 0.05 0.05 0.06            3 3 2 2
```

Segmentation does **not** know what the objects are (greenhouse, pasture…). It only decides which pixels go together. Naming the objects is a later step (classification).

## 3.1 Prepare the input image
Segmentation algorithms compare pixels by their values. Two problems must be solved first:

1. **Bands have different ranges.** The NIR is usually much brighter than the blue. Without rescaling, the NIR would dominate every comparison. We stretch **each band to 0–1** (2nd–98th percentile, as in 2.4).
2. **Array layout.** Our bands are stored as `(band, row, column)`, but `scikit-image` expects `(row, column, band)`. We reorder the axes.

We use **five bands** (blue, green, red, NIR, SWIR1), not only RGB: NIR separates vegetation from plastic, and SWIR helps with bare soil, buildings and water.

**What you should see:** the shape of the input image, e.g. `(1105, 1110, 5)` = rows, columns, 5 bands.

In [ ]:
def normalize_stack(stack, valid, p_low=2, p_high=98):
    '''Stretch each band to 0–1 and reorder the axes for scikit-image.

    stack : array (bands, rows, cols) with reflectance values
    valid : array (rows, cols) of True/False, True = usable pixel
    returns: array (rows, cols, bands) with values between 0 and 1
    '''
    n_bands = stack.shape[0]
    out = np.zeros(stack.shape[1:] + (n_bands,), dtype="float32")   # empty (rows, cols, bands)
    for i, band in enumerate(stack):
        # Percentiles are computed only over valid pixels (clouds would distort them)
        lo, hi = np.nanpercentile(band[valid], [p_low, p_high])
        out[..., i] = np.clip((band - lo) / (hi - lo + 1e-12), 0, 1)
    out[~valid] = 0          # invalid pixels set to 0 (they will be masked anyway)
    return out

# Put the five bands in one 3-D array: (5, rows, cols)
stack = np.stack([arrays[b] for b in ("blue", "green", "red", "nir", "swir16")])
img = normalize_stack(stack, valid)
print(f"✓ Input image for segmentation: {img.shape} (rows, cols, bands), values {img.min():.1f}–{img.max():.1f}")

## 3.2 A single function to run either algorithm
We will run segmentation many times with different settings, so we wrap both algorithms in one function, `run_segmentation()`, that always returns the same kind of result: a label image with **0 = invalid** and **1…N = segments**.

### SLIC (Simple Linear Iterative Clustering)
SLIC is **k-means applied to pixels**, using both their **spectral values** and their **position**:
1. It places cluster centres on a regular grid, separated by a distance `S = √(number of pixels / n_segments)`.
2. Each pixel joins the most similar nearby centre, searching only inside a **2S × 2S window**, which is what makes it fast.
3. Centres are updated and the process repeats (about 10 times).

| Parameter | Meaning | Effect when you increase it |
|---|---|---|
| `n_segments` | Approximate number of segments | Smaller objects |
| `compactness` | Weight of the spatial distance vs. the spectral difference | More regular, square-like segments that follow edges less |

ℹ️ Because our bands are scaled 0–1, useful `compactness` values are small (0.1–2). Tutorials using RGB photos use 10–40 because their colour values are on a 0–100 scale.

### Felzenszwalb (graph-based)
Each pixel is a node connected to its neighbours. Regions are merged when the difference across their border is small compared with the variation already inside them.

| Parameter | Meaning | Effect when you increase it |
|---|---|---|
| `scale` | How easily regions merge | Larger objects |
| `sigma` | Smoothing before segmenting | Less noise, softer edges |
| `min_size` | Minimum segment size, in pixels | Small segments are absorbed |

Felzenszwalb has no option to ignore masked pixels, so we set them to 0 **after** segmenting.

In [ ]:
def run_segmentation(img, valid, method, **params):
    '''Segment 'img' with SLIC or Felzenszwalb.

    img    : array (rows, cols, bands) with values 0–1  (from normalize_stack)
    valid  : array (rows, cols) True/False
    method : "slic" or "felzenszwalb"
    **params : the algorithm parameters, e.g. n_segments=1500, compactness=0.5
    returns: label image (rows, cols) of integers: 0 = invalid, 1..N = segments
    '''
    if method == "slic":
        # channel_axis=-1 → the last axis holds the bands
        # start_label=1   → segments are numbered from 1 (0 is reserved for invalid pixels)
        # mask=valid      → SLIC ignores invalid pixels and labels them 0
        labels = slic(img, channel_axis=-1, start_label=1, mask=valid, **params)
    elif method == "felzenszwalb":
        # Felzenszwalb numbers segments from 0, so we add 1 and then set invalid pixels to 0
        labels = felzenszwalb(img, channel_axis=-1, **params) + 1
        labels[~valid] = 0
    else:
        raise ValueError(f"Unknown method: {method}. Use 'slic' or 'felzenszwalb'.")
    return labels.astype("int32")

print("✓ Function run_segmentation() defined")

## 3.3 A first segmentation
Let us run SLIC once and look at the result. `mark_boundaries()` draws the border of every segment (in yellow) on top of the true-colour image.

**What you should see:** a mesh of yellow lines. Zoom in on a greenhouse: does it fall inside one segment, or is it split into several?

In [ ]:
t0 = time.time()
labels_first = run_segmentation(img, valid, "slic", n_segments=1500, compactness=0.5)
print(f"✓ SLIC produced {labels_first.max()} segments in {time.time() - t0:.1f} s")

# Each segment number is just an ID; the largest number = how many segments there are
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(mark_boundaries(rgb, labels_first, color=(1, 1, 0)))   # yellow borders
ax.set_title("SLIC · n_segments=1500 · compactness=0.5")
ax.axis("off")
plt.show()

## 3.4 Compare algorithms and scales
Now we run **two algorithms at two scales** each and show them side by side. This is the most important figure to understand the **scale parameter**:

- With **few / large** segments → risk of **under-segmentation**: a greenhouse is merged with a road or pasture.
- With **many / small** segments → **over-segmentation**: one greenhouse is split into many pieces.

**What you should see:** four panels. The title of each panel shows the parameters, the number of segments obtained and the running time.

🔧 **Try it:** add a fifth configuration to the `examples` list (and change `plt.subplots(2, 2…)` to `(2, 3…)`), e.g. SLIC with `compactness=2.0`. How does the shape of the segments change?

In [ ]:
# Each entry: (algorithm name, dictionary of parameters)
examples = [
    ("slic",         {"n_segments": 500,  "compactness": 0.5}),   # few, large segments
    ("slic",         {"n_segments": 3000, "compactness": 0.5}),   # many, small segments
    ("felzenszwalb", {"scale": 50,  "sigma": 0.5, "min_size": 20}),  # small scale → small objects
    ("felzenszwalb", {"scale": 800, "sigma": 0.5, "min_size": 20}),  # large scale → large objects
]

fig, axes = plt.subplots(2, 2, figsize=(14, 14))
for ax, (method, params) in zip(axes.ravel(), examples):     # ravel(): 2×2 grid → list of 4 axes
    t0 = time.time()
    labels_i = run_segmentation(img, valid, method, **params)
    ax.imshow(mark_boundaries(rgb, labels_i, color=(1, 1, 0)))
    ax.set_title(f"{method} {params}\n{labels_i.max()} segments · {time.time() - t0:.1f} s", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

### 🔎 Look before you measure
Before computing any metric, write down your visual judgement:
1. In which panel do you see clear **over-segmentation**? Give an example (where?).
2. In which panel do you see **under-segmentation**? Which objects were wrongly merged?
3. Which configuration would you choose *by eye* for greenhouses?

You will compare these answers with the metric in Stage 4.

---
# Stage 4 · Evaluate the segmentation and choose the parameters

Choosing parameters by eye is subjective: two people will choose different values. We use an **unsupervised** metric (no reference polygons needed) proposed by **Johnson & Xie (2011)**. It measures the two properties of a good segmentation:

| Measure | What it checks | High value means… |
|---|---|---|
| **wVar**, weighted variance | Pixels **inside** each segment are similar | **Under-segmentation** (mixed objects) |
| **Moran's I** | **Neighbouring** segments are different from each other | **Over-segmentation** (one object split into similar pieces) |

Both are rescaled 0–1 across all parameter sets tested and added: **Global Score GS = wVar_norm + MoranI_norm**. **Lower GS = better compromise.**

We build the metric in three small functions.

## 4.1 Which segments are neighbours?
Moran's I compares each segment with its **neighbours**, so first we need the list of segment pairs that touch each other.

**How it works:** we compare every pixel with the pixel to its right and the pixel below it. If they have different labels (and neither is 0), those two segments share a border.

```
1 1 2        pixel (1) next to pixel (2) → pair (1, 2)
1 3 2        pixel (1) above pixel (3) → pair (1, 3)
```

In [ ]:
def segment_adjacency(labels):
    '''Return every pair (i, j) of segments that share a border (no repeated pairs).'''
    pairs = []
    # Two comparisons: each pixel with its right neighbour, and with the pixel below
    horizontal = (labels[:, :-1], labels[:, 1:])
    vertical   = (labels[:-1, :], labels[1:, :])
    for a, b in [horizontal, vertical]:
        border = (a != b) & (a > 0) & (b > 0)       # different labels, both valid
        pairs.append(np.stack([a[border], b[border]], axis=1))
    pairs = np.concatenate(pairs)
    # Sort each pair so (2,1) and (1,2) are the same, then remove duplicates
    return np.unique(np.sort(pairs, axis=1), axis=0)

# Quick test on a tiny example
tiny = np.array([[1, 1, 2],
                 [1, 3, 2]])
print("Neighbour pairs in the tiny example:\n", segment_adjacency(tiny))

## 4.2 The Johnson & Xie measures for one segmentation
For a label image and one band (we use **NDVI**), the function computes:

1. **wVar**: for each segment, the variance of its NDVI values; then the average of those variances **weighted by segment area** (big segments count more).
   $$wVar = \frac{\sum_i a_i\, v_i}{\sum_i a_i}$$
2. **Moran's I** of the **segment means**: do neighbouring segments have similar mean NDVI? It uses the neighbour pairs from 4.1 (each neighbour has weight 1).
   $$I = \frac{n}{W}\cdot\frac{\sum_{i}\sum_{j} w_{ij}(\bar{y}_i-\bar{y})(\bar{y}_j-\bar{y})}{\sum_i(\bar{y}_i-\bar{y})^2}$$

We use `pandas.groupby` to compute statistics per segment: it groups all pixels with the same label, like a *dissolve* in GIS but for numbers.

In [ ]:
def johnson_xie(labels, band):
    '''Unsupervised segmentation quality (Johnson & Xie, 2011).

    labels : label image (0 = invalid)
    band   : 2-D array used to measure homogeneity (e.g. NDVI)
    returns: (wVar, MoranI)
    '''
    # --- 1. Statistics per segment ---------------------------------------------
    flat, values = labels.ravel(), band.ravel()        # 2-D → 1-D (one long list of pixels)
    keep = (flat > 0) & np.isfinite(values)            # only valid pixels
    pixels = pd.DataFrame({"segment": flat[keep], "value": values[keep]})
    per_segment = pixels.groupby("segment")["value"]
    area     = per_segment.size()        # number of pixels in each segment
    variance = per_segment.var(ddof=0)   # variance inside each segment
    mean     = per_segment.mean()        # mean value of each segment

    # --- 2. Weighted variance (homogeneity inside segments) ---------------------
    wvar = float((area * variance).sum() / area.sum())

    # --- 3. Moran's I between neighbouring segments -----------------------------
    pairs = segment_adjacency(labels)
    # keep only pairs where both segments have statistics
    ok = np.isin(pairs[:, 0], mean.index) & np.isin(pairs[:, 1], mean.index)
    pairs = pairs[ok]
    deviation = mean - mean.mean()                     # how far each segment is from the global mean
    d_i = deviation.loc[pairs[:, 0]].to_numpy()
    d_j = deviation.loc[pairs[:, 1]].to_numpy()
    n = len(mean)                                      # number of segments
    W = 2 * len(pairs)                                 # sum of weights (each pair counted both ways)
    moran = float((n / W) * (2 * np.sum(d_i * d_j)) / np.sum(deviation.to_numpy() ** 2))
    return wvar, moran

wv, mi = johnson_xie(labels_first, arrays["ndvi"])
print(f"✓ First SLIC segmentation → wVar = {wv:.5f} | Moran's I = {mi:.3f}")

## 4.3 Test many parameter sets and rank them
`evaluate_parameters()` runs the segmentation for every parameter set in a list, computes wVar and Moran's I, rescales both to 0–1 and adds them into the **Global Score (GS)**. The table is sorted so that the **best configuration is in the first row**.

The "parameter grid" is just a list of dictionaries. For SLIC we combine 3 values of `n_segments` × 3 values of `compactness` = 9 segmentations; for Felzenszwalb, 3 values of `scale`.

⏱ This is the slowest cell of the notebook (it segments the image 12 times). It may take one or two minutes.

**What you should see:** two tables. Notice the trade-off: as `n_segments` grows, **wVar goes down** (more homogeneous objects) but **Moran's I goes up** (more similar neighbours).

🔧 **Try it:** the metric is computed on NDVI. If your objects are water, mining or buildings, try another band (e.g. `arrays["swir16"]`).

In [ ]:
def evaluate_parameters(img, valid, band, method, param_grid):
    '''Segment with every parameter set and score it with Johnson & Xie.'''
    rows = []
    for params in param_grid:
        labels_i = run_segmentation(img, valid, method, **params)
        wvar, moran = johnson_xie(labels_i, band)
        rows.append({"method": method, **params,
                     "n_segments_out": int(labels_i.max()),   # segments actually obtained
                     "wVar": wvar, "MoranI": moran})
    table = pd.DataFrame(rows)
    # Min–max rescaling: the smallest value becomes 0 and the largest becomes 1
    rescale = lambda s: (s - s.min()) / (s.max() - s.min() + 1e-12)
    table["GS"] = rescale(table["wVar"]) + rescale(table["MoranI"])   # Global Score
    return table.sort_values("GS").reset_index(drop=True)              # best first

# Parameter grids (lists of dictionaries)
slic_grid = [{"n_segments": n, "compactness": c}
             for n in (500, 1500, 3000)          # 3 values ...
             for c in (0.1, 0.5, 2.0)]           # ... × 3 values = 9 combinations
felz_grid = [{"scale": s, "sigma": 0.5, "min_size": 20} for s in (50, 200, 800)]

t0 = time.time()
eval_slic = evaluate_parameters(img, valid, arrays["ndvi"], "slic", slic_grid)
eval_felz = evaluate_parameters(img, valid, arrays["ndvi"], "felzenszwalb", felz_grid)
print(f"✓ 12 segmentations evaluated in {time.time() - t0:.0f} s")

print("\nSLIC (best first):")
display(eval_slic.round(4))
print("Felzenszwalb (best first):")
display(eval_felz.round(4))

## 4.4 Visualise the trade-off
Each point is one parameter set. The x-axis shows Moran's I (more to the right = more over-segmentation) and the y-axis the weighted variance (higher = more under-segmentation). **The best compromises are close to the bottom-left corner.**

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for table, marker in [(eval_slic, "o"), (eval_felz, "s")]:      # circles = SLIC, squares = Felzenszwalb
    ax.scatter(table["MoranI"], table["wVar"], marker=marker, s=60, label=table["method"].iloc[0])
    # write the number of segments next to each point
    for _, r in table.iterrows():
        ax.annotate(str(r["n_segments_out"]), (r["MoranI"], r["wVar"]),
                    textcoords="offset points", xytext=(5, 4), fontsize=8)
ax.set_xlabel("Moran's I between segments  (over-segmentation →)")
ax.set_ylabel("Weighted variance inside segments  (under-segmentation →)")
ax.set_title("Parameter trade-off (bottom-left is best) · labels = number of segments")
ax.legend()
plt.tight_layout()
plt.show()

## 4.5 Choose the final segmentation
By default we take the **best SLIC configuration** (first row of `eval_slic`).

The metric is a guide, not the final word: it evaluates the **whole image** with **one band**, not specifically your objects. If your visual judgement from Stage 3 strongly disagrees, you may choose another row, but **justify your choice** in the deliverable.

🔧 **Try it:** to use, for example, the second-best row write `best = eval_slic.iloc[1]`. To use Felzenszwalb, change `BEST_METHOD` and build `BEST_PARAMS` with `scale`, `sigma` and `min_size`.

**What you should see:** the chosen parameters and the number of segments, plus the boundaries on the image.

In [ ]:
best = eval_slic.iloc[0]          # first row = lowest Global Score
BEST_METHOD = "slic"
BEST_PARAMS = {"n_segments": int(best["n_segments"]), "compactness": float(best["compactness"])}

labels = run_segmentation(img, valid, BEST_METHOD, **BEST_PARAMS)
print(f"✓ Final segmentation: {BEST_METHOD} {BEST_PARAMS} → {labels.max()} segments")

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(mark_boundaries(rgb, labels, color=(1, 1, 0)))
ax.set_title(f"Final segmentation · {BEST_METHOD} {BEST_PARAMS}")
ax.axis("off")
plt.show()

---
# Stage 5 · Attributes for each object

This is the core idea of GEOBIA: we stop thinking in pixels and describe each **object** with a set of **attributes**. We will obtain a table with **one row per segment**.

| Type | Attributes | Example for greenhouses |
|---|---|---|
| **Spectral** | Mean and standard deviation of each band and of NDVI | Plastic: high blue and red, low NDVI |
| **Size** | Number of pixels, area in m² | 0.5–10 ha |
| **Shape** | Eccentricity, solidity, extent | Compact, rectangular |

**Shape attributes** (computed with `regionprops_table` from scikit-image):
- **Eccentricity** (0–1): 0 = circle; close to 1 = very elongated (roads, rivers).
- **Solidity** (0–1): area ÷ area of the convex hull. 1 = convex shape with no "bays" or holes; low = L-shapes or irregular.
- **Extent** (0–1): area ÷ area of the bounding rectangle. Rectangles aligned with the image are close to 1.

In [ ]:
def object_features(labels, bands, pixel_area_m2):
    '''Compute spectral and shape attributes for each segment.

    labels : label image (0 = invalid)
    bands  : dictionary {name: 2-D array}, e.g. arrays
    pixel_area_m2 : area of one pixel in m² (100 for Sentinel-2 at 10 m)
    returns: DataFrame with one row per segment
    '''
    # --- Spectral attributes: one row per valid pixel, then group by segment ---
    flat = labels.ravel()
    keep = flat > 0
    pixels = pd.DataFrame({"segment_id": flat[keep]})
    for name, arr in bands.items():
        pixels[name] = arr.ravel()[keep]          # one column per band
    groups = pixels.groupby("segment_id")
    feats = groups.mean().add_suffix("_mean").join(groups.std().add_suffix("_std"))

    # --- Size ---
    feats["n_pixels"] = groups.size()
    feats["area_m2"] = feats["n_pixels"] * pixel_area_m2

    # --- Shape (scikit-image measures each labelled region) ---
    shape_attrs = pd.DataFrame(regionprops_table(
        labels, properties=("label", "eccentricity", "solidity", "extent")
    )).rename(columns={"label": "segment_id"}).set_index("segment_id")

    # Join both tables using the segment number as the key
    return feats.join(shape_attrs).reset_index()

feats = object_features(labels, arrays, PIXEL_AREA)
print(f"✓ {len(feats)} objects described with {feats.shape[1]} attributes")
feats.head()      # show the first 5 objects

### Summary of the attributes
`describe()` gives, for each attribute, the number of objects, the mean, the standard deviation, the minimum, the quartiles and the maximum.

**Think about it:** we started with about a million pixels and now have a few thousand objects with dozens of attributes each. What is the typical object area (`area_m2`, 50 % row)? Is it a reasonable size for a greenhouse?

In [ ]:
feats.describe().T.round(3)     # .T transposes: one attribute per row, easier to read

---
# Stage 6 · From raster to vector: segments as polygons

So far our objects are numbers in a raster. To use them in a GIS (QGIS, ArcGIS, PostGIS) we convert them into **polygons**.

`rasterio.features.shapes()` follows the border of each group of pixels with the same label and returns its geometry. It uses the **affine transform** saved in Stage 2 to convert row/column positions into real **UTM coordinates in metres**, so the polygons land in the right place on the map.

A segment may come out as **several pieces**, for example when a cloud mask cuts it in two. `dissolve(by="segment_id")` merges all pieces of the same segment into one record (one row per segment).

**What you should see:** the number of polygons (equal to the number of segments) and the time it took.

In [ ]:
t0 = time.time()

# shapes() yields (geometry as a GeoJSON-like dict, label value) for every patch of pixels.
# mask=labels > 0 → skip invalid pixels (label 0).
polys = [(shape(geom), int(value))
         for geom, value in shapes(labels, mask=labels > 0, transform=TRANSFORM)]

# Build a GeoDataFrame: one column with the segment number + the geometry column
gdf = gpd.GeoDataFrame(
    {"segment_id": [value for _, value in polys]},
    geometry=[geom for geom, _ in polys],
    crs=CRS,                                  # same CRS as the image (UTM, metres)
)
gdf = gdf.dissolve(by="segment_id").reset_index()     # merge pieces of the same segment

print(f"✓ {len(gdf)} polygons created in {time.time() - t0:.1f} s | CRS: {gdf.crs}")

## 6.1 Add a vector shape index and join the attributes
Now that objects are polygons we can measure their **perimeter** and compute the **Polsby–Popper compactness index**:

$$PP = \frac{4\pi \cdot \text{area}}{\text{perimeter}^2}$$

It is 1 for a perfect circle and decreases for irregular or elongated shapes. Polygons traced from pixels have "staircase" borders, so their perimeter is overestimated and PP is lower than the "true" value. Use it to **compare objects with each other**, not as an absolute value.

Then we **join** the attribute table from Stage 5 using `segment_id` as the key (like a table join in QGIS), and we add the scene ID and date so we always know where the data came from.

**What you should see:** the first rows of the final table: geometry + all attributes.

In [ ]:
gdf["perimeter_m"] = gdf.geometry.length                                # in metres (UTM)
gdf["polsby_popper"] = 4 * np.pi * gdf.geometry.area / gdf["perimeter_m"] ** 2

# Table join: add the spectral and shape attributes computed from the raster
gdf = gdf.merge(feats, on="segment_id")

# Metadata: which scene and date these objects come from (good practice!)
gdf["scene_id"] = item.id
gdf["acq_date"] = item.datetime.date().isoformat()

print(f"✓ Final table: {len(gdf)} objects × {gdf.shape[1]} columns")
gdf.head()

## 6.2 Quick check: colour the objects by mean NDVI
A simple map to verify that the polygons are in the right place and that the attributes make sense: greenhouses should appear with low NDVI (red/orange), pastures with high NDVI (green).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
gdf.plot(column="ndvi_mean", cmap="RdYlGn", vmin=-0.2, vmax=0.9,
         edgecolor="black", linewidth=0.1, legend=True,
         legend_kwds={"label": "Mean NDVI per object", "shrink": 0.6}, ax=ax)
ax.set_title("Image objects coloured by mean NDVI")
ax.set_axis_off()
plt.show()

---
# Stage 7 · Save the objects as GeoParquet (Week 5 recap)

We save the same objects in two formats and compare them:

| | GeoParquet | GeoJSON |
|---|---|---|
| Storage | Binary, **columnar**, compressed | Text (JSON), row by row |
| CRS | Any; stored in the `geo` metadata of the file footer | Must be **WGS 84** (RFC 7946), so we reproject first |
| Reading part of the data | Can read **only some columns** | Must read the whole file |

## 7.1 Write both files

In [ ]:
pq_path = os.path.join(OUTPUT_DIR, f"{AOI_NAME}_objects.parquet")
gj_path = os.path.join(OUTPUT_DIR, f"{AOI_NAME}_objects.geojson")

gdf.to_parquet(pq_path)                                   # keeps the UTM CRS
gdf.to_crs(4326).to_file(gj_path, driver="GeoJSON")       # GeoJSON requires WGS 84

print(f"✓ Saved: {pq_path}")
print(f"✓ Saved: {gj_path}")

## 7.2 Compare file sizes and read only some columns
**What you should see:** GeoParquet is usually several times smaller than GeoJSON. Then we read back **only 4 of the columns**. With large datasets this column selection (*column projection*, Week 5) saves a lot of time and memory.

In [ ]:
size_pq = os.path.getsize(pq_path) / 1e6     # bytes → megabytes
size_gj = os.path.getsize(gj_path) / 1e6
print(f"GeoParquet: {size_pq:6.2f} MB")
print(f"GeoJSON   : {size_gj:6.2f} MB  → GeoParquet is {size_gj / size_pq:.1f}× smaller")

# Read only the columns we need (the rest of the file is not even read)
subset = gpd.read_parquet(pq_path, columns=["segment_id", "ndvi_mean", "area_m2", "geometry"])
print(f"✓ Read back {subset.shape[1]} of {gdf.shape[1]} columns | CRS preserved: {subset.crs}")
subset.head()

---
# Stage 8 · A first grouping of objects with k-means (preview of next week)

In the warm-up you practised **k-means** with points and with the colours of a photo. Here we apply the **same algorithm to our objects**: each object is a point described by its attributes (NDVI, reflectances, shape…), and k-means groups objects that are similar.

This is **unsupervised**: k-means finds groups but does **not** know their names. It is up to you to interpret which group corresponds to greenhouses. Next week we will train a **supervised** classifier with labelled examples.

## 8.1 Standardise the attributes and run k-means
k-means uses **distances**. If one attribute has much bigger numbers than the others (e.g. area in m² vs. reflectance between 0 and 1), it dominates the distance. `StandardScaler` transforms every attribute to **mean 0 and standard deviation 1**, so all attributes weigh the same.

| Parameter | Meaning |
|---|---|
| `n_clusters=6` | Number of groups to find (you choose it, as in the warm-up) |
| `n_init=10` | Run k-means 10 times from different starting points and keep the best result |
| `random_state=42` | Fix the random starting points so the result is reproducible |

🔧 **Try it:** change `n_clusters` (4, 8, 10) or add/remove attributes from `FEATURES`. Does a group appear that matches your objects better?

In [ ]:
# Attributes used to compare objects (spectral + shape)
FEATURES = ["ndvi_mean", "ndvi_std", "red_mean", "nir_mean", "swir16_mean",
            "blue_mean", "polsby_popper", "solidity", "eccentricity"]

# fillna(0): replace possible missing values (e.g. std of a 1-pixel segment) before scaling
X = StandardScaler().fit_transform(gdf[FEATURES].fillna(0))

kmeans = KMeans(n_clusters=6, n_init=10, random_state=42)
gdf["cluster"] = kmeans.fit_predict(X).astype(str)    # as text, so maps treat it as a category

print("✓ Number of objects in each cluster:")
print(gdf["cluster"].value_counts().sort_index())

## 8.2 Interpret the clusters
The table shows the **average attributes of each cluster**. Look for the cluster that matches your objects. For greenhouses: **low NDVI, high blue reflectance and compact shapes** (high `polsby_popper`).

In [ ]:
summary = gdf.groupby("cluster")[["ndvi_mean", "blue_mean", "swir16_mean", "area_m2", "polsby_popper"]].mean()
summary["n_objects"] = gdf["cluster"].value_counts()
summary.round(3)

## 8.3 Interactive map
`.explore()` creates an interactive map (Week 5) on top of a satellite basemap. **Click** on an object to see its attributes; **hover** to see the tooltip.

**What you should see:** objects coloured by cluster. Zoom in on the greenhouses: do they all fall in the same cluster?

In [ ]:
gdf.to_crs(4326).explore(                  # web maps work in WGS 84
    column="cluster",                      # colour by cluster
    categorical=True,                      # clusters are categories, not numbers
    cmap="tab10",                          # 10 distinct colours
    tooltip=["segment_id", "cluster", "ndvi_mean", "area_m2", "polsby_popper"],
    style_kwds=dict(weight=0.3, fillOpacity=0.5),   # thin borders, semi-transparent fill
    tiles="Esri.WorldImagery",             # satellite basemap to compare with reality
)

---
# ✅ Your turn: deliverables

Adapt the notebook to **your own area** (cell 0.3) and submit:

1. **Area and object:** your `BBOX`, `DATE_RANGE` and the object type you want to delineate, with one sentence explaining why it suits object-based analysis at 10 m (is it large enough compared with a pixel?).
2. **Segmentation:** at least **2 algorithms × 3 parameter values**, shown as boundary overlays (Stage 3.4, extended).
3. **Evaluation:** the Johnson & Xie tables and plot (Stage 4) and your final choice. Does the metric agree with your visual judgement from Stage 3? If not, which one do you trust and why?
4. **Objects:** your `*_objects.parquet` file and the interactive map (Stage 8.3).
5. **Reflection (half a page):** which cluster(s) correspond to your objects? Which attributes separate them best? What would you need to turn this into a supervised classification?

### Common problems
| Problem | Likely cause | Solution |
|---|---|---|
| `No scenes found` | Filters too strict, or area on a tile edge | Widen `DATE_RANGE`, raise `MAX_CLOUD`, move the `BBOX` |
| Very few valid pixels | The scene is cloudy over your area | In 1.2 use `item = items[1]` (next scene) |
| The kernel dies / out of memory | `BBOX` too large | Go back to about 10 × 10 km |
| SLIC returns very few segments | `compactness` too low | Use values between 0.5 and 2 |
| Square segments that ignore edges | `compactness` too high | Lower it |
| Error writing GeoParquet | `pyarrow` missing | `pip install pyarrow` |
| The interactive map does not appear | `folium` / `mapclassify` missing | `pip install folium mapclassify` |

### References
- Blaschke, T. (2010). Object based image analysis for remote sensing. *ISPRS J. Photogramm. Remote Sens.*, 65(1), 2–16.
- Achanta, R., et al. (2012). SLIC superpixels compared to state-of-the-art superpixel methods. *IEEE TPAMI*, 34(11), 2274–2282.
- Felzenszwalb, P. F., & Huttenlocher, D. P. (2004). Efficient graph-based image segmentation. *IJCV*, 59(2), 167–181.
- Johnson, B., & Xie, Z. (2011). Unsupervised image segmentation evaluation and refinement using a multi-scale approach. *ISPRS J. Photogramm. Remote Sens.*, 66(4), 473–483.
- VanderPlas, J. *In Depth: k-Means Clustering*. Python Data Science Handbook.